# Faza 6 — provera rezultata i robustnost ulaza


## Goal

Ponovo izračunaj baseline test metrike iz `best.pt`, potvrdi ih prema Phase 5
rezultatu i zatim, bez dodatnog treninga, izmeri uticaj niže rezolucije, blur-a
i kontrolisanog crop pomeranja. Sve varijante koriste isti test split, isti
checkpoint, isti greedy CTC dekoder i corpus-level WER/CER.


## Setup

Izaberi T4 GPU. Pre ovog notebooka mora biti završen novi length-aware notebook
Faze 5 i na Drive-u moraju postojati `phase5_length_aware_v2/best.pt` i
`phase5_length_aware_v2/results.json`.


In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'editdistance>=0.8.1', 'opencv-python-headless>=4.10'],
    check=True,
)


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/nikolabakic/Vizuelno-prepoznavanje-govora-na-osnovu-pokreta-usana-pomo-u-LipNet-modela.git'
REPO = Path('/content/lipnet-serbian')
if not (REPO / 'lipnet').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)

# A git pull updates source files, but an active Colab kernel can still hold
# old lipnet modules in sys.modules.  Clear only this project's modules so
# all later imports use one coherent checkout.
stale_modules = [
    name for name in sys.modules
    if name == 'lipnet' or name.startswith('lipnet.')
]
for name in stale_modules:
    del sys.modules[name]
repo_path = str(REPO)
sys.path[:] = [entry for entry in sys.path if entry != repo_path]
sys.path.insert(0, repo_path)
importlib.invalidate_caches()

print('Repo:', REPO)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
if stale_modules:
    print('Osveženi Python moduli:', ', '.join(sorted(stale_modules)))


In [ ]:
import hashlib
import inspect
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from lipnet.model import LipNet

forward_parameters = inspect.signature(LipNet.forward).parameters
assert 'lengths' in forward_parameters, (
    'Učitana LipNet klasa nije length-aware. Ponovo pokreni Setup ćelije; '
    f'model={inspect.getfile(LipNet)}, signature={inspect.signature(LipNet.forward)}'
)
assert torch.cuda.is_available(), 'Uključi T4 GPU u Colab Runtime postavkama.'
DEVICE = torch.device('cuda')
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print('LipNet:', inspect.getfile(LipNet), inspect.signature(LipNet.forward))
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/LipNet')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive izlaz:', DRIVE_ROOT)


In [ ]:
# Faza 7 moze da se pokrene pre nego sto su novi decoder moduli objavljeni
# na GitHub-u. Kopije na Drive-u su mali izvorni fajlovi; mouth frejmovi se
# samo citaju iz postojeceg ZIP-a i nikada se ponovo ne generisu.
import shutil

PHASE7_CODE_DIR = DRIVE_ROOT / 'phase7_code'
for module_name in ('decoder.py', 'evaluation.py'):
    drive_source = PHASE7_CODE_DIR / module_name
    repo_target = REPO / 'lipnet' / module_name
    if drive_source.exists():
        shutil.copy2(drive_source, repo_target)
        print('Ucitan Faza 7 modul sa Drive-a:', drive_source)
    assert repo_target.exists(), (
        f'Nedostaje {module_name}. Ocekivan je u checkout-u ili u '
        f'{PHASE7_CODE_DIR}.'
    )
importlib.invalidate_caches()


## Steps

### 1. Učitaj test mouth frejmove i anotacije


In [ ]:
import zipfile

MOUTH_ARCHIVE = DRIVE_ROOT / 'ai_speak_lip.zip'
SOURCE_ARCHIVE = Path('/content/drive/MyDrive/processed.zip')  # promeni po potrebi
MOUTH_ROOT = Path('/content/ai_speak_lip')
ALIGN_EXTRACT = Path('/content/ai_speak_align')
assert MOUTH_ARCHIVE.exists(), MOUTH_ARCHIVE
assert SOURCE_ARCHIVE.exists(), SOURCE_ARCHIVE

if not next(MOUTH_ROOT.glob('spk*/video/video_a/*'), None):
    MOUTH_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(MOUTH_ARCHIVE) as archive:
        archive.extractall(MOUTH_ROOT)
if not next(ALIGN_EXTRACT.rglob('spk*/alignment/*.align'), None):
    ALIGN_EXTRACT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
        members = [
            member for member in archive.infolist()
            if '/alignment/' in f'/{member.filename}'
            and member.filename.endswith('.align')
        ]
        archive.extractall(ALIGN_EXTRACT, members=members)
CORPUS_ROOT = next(ALIGN_EXTRACT.rglob('spk*/alignment/*.align')).parents[2]


In [ ]:
from torch.utils.data import DataLoader, Subset

from data.splits import TEST_SPEAKERS
from lipnet.dataset import SerbianDataset, variable_length_collate
from lipnet.train import scan_ctc_compatibility

raw_test_dataset = SerbianDataset(
    MOUTH_ROOT, CORPUS_ROOT, TEST_SPEAKERS, phase='test'
)
ctc_report = scan_ctc_compatibility(raw_test_dataset)
test_dataset = Subset(raw_test_dataset, ctc_report.valid_indices)
test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    collate_fn=variable_length_collate,
    pin_memory=True,
)
print('Test:', len(test_dataset), 'CTC-odbačeno:', ctc_report.invalid_count)


### 2. Učitaj tačno provereni Phase 5 checkpoint


In [ ]:
from lipnet.dataset import SERBIAN_LETTERS
from lipnet.model import LipNet

PHASE5_DIR = DRIVE_ROOT / 'phase5_length_aware_v2'
BEST_CHECKPOINT = PHASE5_DIR / 'best.pt'
PHASE5_RESULTS = PHASE5_DIR / 'results.json'
EXPERIMENT_RESULTS = PHASE5_DIR / 'robustness_results.json'
EXPERIMENT_PLOT = PHASE5_DIR / 'robustness_metrics.png'
assert BEST_CHECKPOINT.exists() and PHASE5_RESULTS.exists()

checkpoint = torch.load(BEST_CHECKPOINT, map_location='cpu', weights_only=False)
assert checkpoint['metadata']['training_protocol'] == 'length-aware-bigru-corpus-metrics-v2'
model = LipNet(num_classes=1 + len(SERBIAN_LETTERS)).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model.eval()

hasher = hashlib.sha256()
with BEST_CHECKPOINT.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b''):
        hasher.update(chunk)
checkpoint_sha256 = hasher.hexdigest()
phase5_results = json.loads(PHASE5_RESULTS.read_text(encoding='utf-8'))
assert phase5_results['best_checkpoint_sha256'] == checkpoint_sha256
assert phase5_results['metrics_definition'] == 'corpus-edit-distance-v1'


### 3. Definiši determinističke ulazne varijante


In [ ]:
def map_frames(video, operation):
    batch, channels, time, height, width = video.shape
    frames = video.permute(0, 2, 1, 3, 4).reshape(
        batch * time, channels, height, width
    )
    changed = operation(frames)
    return changed.reshape(batch, time, channels, height, width).permute(0, 2, 1, 3, 4)

def identity(video):
    return video

def resolution(video, height, width):
    def resize(frames):
        low = F.interpolate(
            frames, size=(height, width), mode='bilinear', align_corners=False
        )
        return F.interpolate(
            low, size=(64, 128), mode='bilinear', align_corners=False
        )
    return map_frames(video, resize)

def gaussian_blur(video, kernel_size=5, sigma=1.0):
    coordinates = torch.arange(kernel_size, device=video.device) - kernel_size // 2
    kernel_1d = torch.exp(-(coordinates.float() ** 2) / (2 * sigma**2))
    kernel_1d /= kernel_1d.sum()
    kernel_2d = torch.outer(kernel_1d, kernel_1d)
    kernel = kernel_2d.expand(3, 1, kernel_size, kernel_size)
    def blur(frames):
        padded = F.pad(
            frames,
            (kernel_size // 2,) * 4,
            mode='reflect',
        )
        return F.conv2d(padded, kernel, groups=3)
    return map_frames(video, blur)

def crop_shift(video, dx=4, dy=2):
    shifted = torch.zeros_like(video)
    shifted[:, :, :, dy:, dx:] = video[:, :, :, :-dy, :-dx]
    return shifted

conditions = {
    'baseline_128x64': identity,
    'resolution_96x48': lambda video: resolution(video, 48, 96),
    'resolution_64x32': lambda video: resolution(video, 32, 64),
    'gaussian_blur_5_sigma1': gaussian_blur,
    'crop_shift_dx4_dy2': crop_shift,
}


### 4. Izmeri sve uslove istim dekoderom i definicijom metrika


In [ ]:
from lipnet.train import greedy_decode, reference_text, sequence_metrics

condition_outputs = {}
for name, transform in conditions.items():
    predictions = []
    references = []
    with torch.inference_mode():
        for batch in test_loader:
            video = transform(batch['vid'].to(DEVICE, non_blocking=True))
            logits = model(video, lengths=batch['vid_len'])
            predictions.extend(
                greedy_decode(logits, output_lengths=batch['vid_len'])
            )
            references.extend(reference_text(batch))
    metrics = sequence_metrics(predictions, references)
    condition_outputs[name] = {
        'metrics': metrics,
        'predictions': predictions,
        'references': references,
    }
    print(name, json.dumps(metrics, ensure_ascii=False))

baseline_metrics = condition_outputs['baseline_128x64']['metrics']
for metric in ('wer', 'cer', 'sentence_exact_match'):
    assert math.isclose(
        baseline_metrics[metric],
        phase5_results['test'][metric],
        rel_tol=0.0,
        abs_tol=1e-12,
    ), (metric, baseline_metrics[metric], phase5_results['test'][metric])
print('PASS: baseline se tačno poklapa sa sačuvanom Phase 5 evaluacijom.')


## Checks

### 5. Sačuvaj delte, kvalitativne greške i grafikon


In [ ]:
summary = {}
for name, output in condition_outputs.items():
    metrics = output['metrics']
    errors = [
        {'reference': reference, 'prediction': prediction}
        for reference, prediction in zip(output['references'], output['predictions'])
        if reference != prediction
    ][:10]
    summary[name] = {
        **metrics,
        'wer_delta_vs_baseline': metrics['wer'] - baseline_metrics['wer'],
        'cer_delta_vs_baseline': metrics['cer'] - baseline_metrics['cer'],
        'qualitative_errors': errors,
    }

payload = {
    'phase': 6,
    'checkpoint_sha256': checkpoint_sha256,
    'metrics_definition': 'corpus-edit-distance-v1',
    'test_samples': len(test_dataset),
    'environment': {
        'torch_version': torch.__version__,
        'cuda_version': torch.version.cuda,
        'gpu': torch.cuda.get_device_name(0),
    },
    'conditions': summary,
}
EXPERIMENT_RESULTS.write_text(
    json.dumps(payload, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)

names = list(summary)
figure, axes = plt.subplots(1, 2, figsize=(13, 4))
for axis, metric in zip(axes, ('wer', 'cer')):
    values = [summary[name][metric] for name in names]
    axis.barh(names, values)
    axis.set_xlabel(metric.upper())
    axis.set_xlim(left=0)
    axis.grid(axis='x', alpha=0.25)
    for index, value in enumerate(values):
        axis.text(value, index, f' {value:.4f}', va='center')
plt.tight_layout()
figure.savefig(EXPERIMENT_PLOT, dpi=160, bbox_inches='tight')
plt.show()
print('Sačuvano:', EXPERIMENT_RESULTS)
print('Sačuvano:', EXPERIMENT_PLOT)


## Next Steps

Rezultate tumači tek nakon uspešnog baseline `assert` poređenja. Veći pozitivni
`wer_delta_vs_baseline` znači da je intervencija pogoršala model. U završni tekst
prenesi samo vrednosti iz `robustness_results.json` i navedi checkpoint SHA-256,
jer time ostaje proverljivo koji je model evaluiran.
